# АНАЛИЗ СУЩЕСТВУЮЩИХ РЕШЕНИЙ (СИСТЕМ-АНАЛОГОВ)

## 1. ВВЕДЕНИЕ

### 1.1. Цель анализа

Целью анализа является сопоставление трех нейросетевых архитектур — HyenaPixel, Mamba и UNet++ — как возможной основы для систем анализа пространственных данных в строительстве: спутниковых и аэрофотоснимков строительных площадок, BIM‑чертежей, планов этажей, изображений с камер наблюдения на объектах. В рамках анализа рассматривается, как каждая из архитектур масштабируется по пространству, какие показатели качества и производительности демонстрирует в задачах семантической сегментации и какие конкурентные преимущества может получить разрабатываемый проект при выборе той или иной архитектуры в качестве базовой.

### 1.2. Критерии отбора систем для анализа

Для анализа выбраны следующие архитектуры:

- **HyenaPixel** — вариант архитектуры для обработки двумерных изображений, в которой вместо механизма внимания используются очень большие сверточные ядра семейства Hyena; архитектура представляет интерес как эффективная базовая сеть для детекции и сегментации изображений высокого разрешения.
- **Mamba** — универсальная архитектура линейной по длине последовательности сложности на основе селективных моделей в пространстве состояний; рассматривается для моделирования больших пространственно‑временных последовательностей.
- **UNet++** — развитая U‑образная архитектура для семантической и объектной сегментации с вложенными связями между уровнями; уже широко применяется в медицинской и городской сегментации и имеет готовые реализации в популярных фреймворках, в том числе частично в экосистеме MMSegmentation.

Основные критерии выбора:

- возможность непосредственно работать с двумерными пространственными данными или быть относительно просто адаптируемой к ним;
- масштабируемость по размеру входных изображений, что критично для строительных задач с крупными планами и снимками высокого разрешения.

## 2. ИДЕНТИФИКАЦИЯ СИСТЕМ-АНАЛОГОВ

| № | Название   | Разработчик / статья      | Тип модели                        | Основной домен применения          |
|---|------------|---------------------------|-----------------------------------|-------------------------------------|
| 1 | HyenaPixel | Spravil et al., "HyenaPixel" | 2D‑архитектура на базе Hyena в каркасе MetaFormer | Задачи компьютерного зрения, детекция, сегментация, анализ сцен высокого разрешения |
| 2 | Mamba      | Gu et al., "Mamba"       | Селективная модель в пространстве состояний | Модели языка, аудио, геномика, длинные временные и пространственно‑временные последовательности |
| 3 | UNet++     | Zhou et al., "UNet++"    | U‑образная сеть с вложенными пропусками | Медицинская и городская семантическая сегментация |

## 3. ФУНКЦИОНАЛЬНЫЙ СРАВНИТЕЛЬНЫЙ АНАЛИЗ

### Таблица 1. Сравнение функциональных возможностей

Легенда: `++` — функциональность реализована полноценно и типична для архитектуры; `+` — функциональность возможна или поддерживается частично; `-` — сценарий не является целевым «из коробки».

| Функция / сценарий | HyenaPixel | Mamba | UNet++ |
|--------------------|-----------|-------|--------|
| Семантическая сегментация двумерных сцен (планы, ортофотоснимки) | ++ | + (через развёртку 2D→1D) | ++ |
| Instance‑ и паноптическая сегментация объектов (краны, техника, этажи) | ++ (через детектор и соответствующую голову сегментации) | + | ++ |
| Работа с очень высокими разрешениями (4K+ ортофотоснимки строительной площадки) | ++ (субквадратичная сложность по числу пикселей) | + (линейная сложность по длине последовательности, но использование памяти зависит от способа развёртки) | + (ограничения, характерные для классического U‑Net) |
| Учёт контекста на уровне всей сцены (городской квартал, строительный кластер) | ++ (очень большое эффективное рецептивное поле) | ++ (длинный контекст по последовательности) | + (ограничен глубиной сети и размером рецептивного поля) |
| Пространственно‑временной анализ (эволюция строительства по серии снимков) | + | ++ (естественная область применения для SSM) | + (через ConvLSTM или трёхмерные модификации) |
| Наличие готовых реализаций и интеграций в mmseg/mmcv | Частично ( через собственную реализацию базовой сети) | Отсутствуют готовые реализации, требуется адаптация | Есть готовые U‑Net‑подобные архитектуры, UNet++ несложно реализовать в существующей экосистеме |

## 4. ДЕТАЛЬНАЯ ХАРАКТЕРИСТИКА СИСТЕМ-АНАЛОГОВ

### 4.1. HyenaPixel

#### 4.1.1. Общее описание

HyenaPixel расширяет одномерный оператор Hyena на двунаправленные одномерные последовательности, а затем и на двумерные изображения, внедряя двумерный вариант Hyena в каркас MetaFormer в качестве механизма смешивания токенов вместо внимания. Ключевая идея заключается в использовании свёрточных ядер, размер которых может превышать размер карты признаков (до 191×191), что обеспечивает видимость всего кадра для каждого пикселя и формирование глобального контекста.

#### 4.1.2. Основные функциональные возможности

- Формирование глобального двумерного контекста для изображений за счёт свёрток очень большого размера при субквадратичной сложности по числу пикселей.
- Конкурентоспособная точность на датасете ImageNet‑1k (до 84,9 % top‑1 без использования внешних данных).
- Успешное применение в задачах детекции и семантической/объектной сегментации (MS COCO, ADE20K) в роли базовой сети в стандартных архитектурах детекции и сегментации.
- Естественная применимость к анализу крупных планов и ортофотоснимков строительных объектов, где одновременно важны как мелкие детали (арматура, строительные леса), так и глобальный контекст (границы площадки, соседние здания).

#### 4.1.3. Технические характеристики

- **Тип архитектуры:** базовая сеть формата MetaFormer с использованием HyenaPixel в качестве модуля смешивания токенов.
- **Механизм смешивания токенов:** двумерный Hyena — иерархия длинных свёрток с механизмом «gating», развивающая одномерную архитектуру Hyena Hierarchy.
- **Сложность:** субквадратичная по числу пикселей за счёт применения свёрток вместо квадратичного по размеру входа механизма внимания.
- **Рецептивное поле:** эффективное рецептивное поле охватывает весь входной кадр.
- **Интеграция:** существует реализация на PyTorch от авторов, совместимая с типовыми фреймворками компьютерного зрения; возможна интеграция в MMSegmentation через реализацию собственной базовой сети.

#### 4.1.4. Преимущества

- Очень большое эффективное рецептивное поле и хорошая локализация объектов на изображениях высокого разрешения, что особенно полезно для задач разметки строительных площадок, дорожной сети и зданий.
- Субквадратичная сложность позволяет использовать более высокое разрешение входных изображений при фиксированном ресурсе графического процессора по сравнению с архитектурами типа ViT с полным механизмом внимания.
- Возможность использовать HyenaPixel как взаимозаменяемую базовую сеть вместо трансформерных архитектур в уже существующих конвейерах детекции и сегментации (например, Mask R‑CNN, Cascade).

#### 4.1.5. Недостатки

- Архитектура ориентирована прежде всего на двумерные изображения; для пространственно‑временных задач строительства (учёт динамики во времени) требуется дополнительное расширение (трёхмерный или временной вариант Hyena).
- Для практического применения в строительной отрасли необходимо внедрение архитектуры в MMSegmentation или аналогичный инструмент, что потребует дополнительной инженерной работы (описание базовой сети и конфигураций).

#### 4.1.6. Метрики качества (семантическая сегментация)

В работе, посвящённой HyenaPixel, показано, что при использовании данной архитектуры в качестве базовой сети в стандартных головах сегментации достигается конкурентный средний показатель mIoU на наборе ADE20K и высокое качество по mAP на COCO‑детекции; конкретные значения зависят от глубины модели, размеров ядер и разрешения входа. В контексте MMSegmentation для оценки таких моделей используется стандартный набор метрик:

- **aAcc** — общая точность по пикселям;
- **mAcc** — средняя точность по классам;
- **mIoU** — средний Intersection over Union по классам;
- **mDice** — средний коэффициент Dice;
- **mFscore**, **mPrecision**, **mRecall**.

Для строительных задач эти метрики особенно важны: например, mIoU по классам «здание», «дорога», «строительная техника», «опасные зоны» может напрямую отразить пригодность модели для задач безопасного планирования и мониторинга строительной площадки.

### 4.2. Mamba

#### 4.2.1. Общее описание

Mamba представляет собой архитектуру для обработки последовательностей с линейной сложностью по длине, основанную на селективных моделях в пространстве состояний, где параметры модели зависят от входного сигнала и позволяют избирательно запоминать наиболее важную информацию. Модель демонстрирует качество, сопоставимое с трансформерными архитектурами на языковых, звуковых и геномных задачах, при существенно меньшей вычислительной сложности и более высокой пропускной способности.

#### 4.2.2. Основные функциональные возможности

- Линейная по длине контекста сложность с применением параллельной операции сканирования, учитывающей особенности аппаратной реализации, что обеспечивает ускорение до пяти раз по сравнению с механизмом самовнимания при сопоставимом уровне качества.
- Возможность моделировать чрезвычайно длинные последовательности (вплоть до миллионов токенов) без квадратичного роста потребления памяти.
- Универсальность по типам данных: модель применяется к тексту, аудио и временным рядам, включая данные с датчиков на строительных объектах.

Для пространственных задач это позволяет:

- рассматривать серии ортофотоснимков строительной площадки как временную последовательность и моделировать динамику;
- обрабатывать траектории сканирования лидаром или профили высот в виде длинных одномерных последовательностей;

#### 4.2.3. Технические характеристики

- **Тип архитектуры:** блоки SSM‑типа S6 с селективным обновлением состояния и одномерной свёрткой на входе.
- **Механизм смешивания токенов:** селективная модель в пространстве состояний вместо внимания; архитектура формально рекуррентная, но реализуемая в параллельном режиме с помощью операции сканирования.
- **Сложность:** O(L) по длине последовательности с высокой пропускной способностью на современных графических процессорах.
- **Интеграция:** на данный момент отсутствует готовая реализация Mamba в качестве базовой сети в MMSegmentation, однако последовательностная природа модели позволяет развёртывать двумерные карты признаков в одномерную последовательность по аналогии с ViT и комбинировать Mamba с классическими головами сегментации.

#### 4.2.4. Преимущества

- Хорошо подходит для пространственно‑временных сценариев (совместная обработка временных рядов и изображений), которые особенно актуальны в строительстве: мониторинг прогресса работ, анализ соблюдения графика, прогнозирование рисков.
- Линейная сложность делает возможной обработку как длинных историй, так и очень крупных карт при аккуратной развёртке без квадратичных затрат, характерных для самовнимания.
- Возможность построения мультимодальных систем, в которых строительные планы, текстовая документация и временные ряды рассматриваются в единой рамке моделей в пространстве состояний.

#### 4.2.5. Недостатки

- Отсутствие нативной двумерной структуры: модель исходит из представления данных в виде одномерной последовательности, поэтому восстановление локальной двумерной организации требует аккуратного позиционного кодирования и проектирования схем развёртки.
- Нет готовой интеграции в MMSegmentation; реализация базовой сети, декодера, конфигураций и подключение стандартных метрик (mIoU, mDice и др.) требуют существенных инженерных усилий.

#### 4.2.6. Метрики и пригодность для сегментации

При переносе Mamba в задачи сегментации по аналогии с визуальными моделями используются те же метрики, что и в MMSegmentation: mIoU, mDice, Pixel Accuracy, mPrecision, mRecall и др. Это позволяет применять Mamba как ядро модели для:

- оценки качества сегментации прогресса строительства по снимкам (например, mIoU по классам «завершённые конструкции», «в процессе», «не начато»);
- измерения mDice и mFscore для тонких и редких объектов (арматура, балки, строительные леса).

### 4.3. UNet++

#### 4.3.1. Общее описание

UNet++ представляет собой модификацию классического U‑Net, в которой введены вложенные и плотные связи между энкодером и декодером, а также используется механизм обучения с контролем на нескольких уровнях, что фактически образует ансамбль U‑Net‑подобных сетей разной глубины в рамках одной архитектуры. Целью таких модификаций является более полное использование признаков разных масштабов и улучшение сегментации объектов различного размера.

#### 4.3.2. Основные функциональные возможности

- Высокое качество семантической и объектной сегментации на различных наборах медицинских и иных изображений за счёт мультимасштабной агрегации признаков.
- Улучшенная сегментация как мелких, так и крупных объектов по сравнению с базовым U‑Net, что критично для строительных задач (мелкие элементы арматуры, окна, двери, а также крупные блоки зданий).
- Гибкость: UNet++ может использовать различные сверточные базовые сети и сравнительно просто интегрируется в существующие конвейеры обработки, включая те, что реализованы в MMSegmentation.

#### 4.3.3. Технические характеристики

- **Тип архитектуры:** полноcверточный энкодер–декодер с вложенными пропусками (dense skip‑связями), образующими серию декодеров разной глубины при общем энкодере.
- **Сложность:** сопоставима с U‑Net аналогичной глубины; возможна последующая «обрезка» (pruning) для ускорения вывода при умеренном снижении качества.
- **Рецептивное поле:** определяется глубиной энкодера и размерами свёрток; вложенные связи обеспечивают более эффективное использование мультимасштабных признаков, но глобальный контекст формируется менее эффективно, чем в HyenaPixel или Mamba.

#### 4.3.4. Применение к городским и строительным сценам

UNet‑подобные модели уже широко применяются для сегментации городских сцен (например, Cityscapes) и демонстрируют высокие значения общей точности, mIoU и среднего коэффициента Dice при разметке дорог, зданий, людей и транспорта. Исследования показывают, что ансамбли U‑Net‑ и UNet‑подобных архитектур на Cityscapes достигают, в частности, точности порядка 87,6 % и mIoU около 0,65 для многоклассовой сегментации городской сцены, что является хорошей отправной точкой для задач сегментации строительных объектов.

В официальном репозитории MMSegmentation основное внимание уделяется базовым U‑Net и более современным базовым архитектурам, однако UNet++ несложно реализовать как пользовательскую модель с использованием доступных компонентов и оценивать с помощью стандартных метрик (mIoU, mDice, mFscore и др.).

#### 4.3.5. Метрики (в стиле MMSegmentation)

MMSegmentation предоставляет класс `IoUMetric`, который по умолчанию вычисляет следующие показатели:

- **aAcc** — общая точность по пикселям;
- **mAcc** — средняя точность по классам;
- **mIoU** — средний показатель IoU по классам;
- **mDice** — средний коэффициент Dice;
- **mFscore**, **mPrecision**, **mRecall**.

В типичных экспериментах по сегментации (включая городские сцены) UNet++ демонстрирует высокие значения точности, mIoU и среднего коэффициента Dice, превосходя базовый U‑Net и FCN. Для строительных задач могут использоваться те же метрики и их поклассные значения (per‑class IoU/Dice для классов «здание», «дорога», «кран», «ограждение», «опасная зона» и др.).

## 5. SWOT-АНАЛИЗ

### HyenaPixel

- **S (Strengths, сильные стороны):**  
  - Очень большое эффективное рецептивное поле и глобальный контекст при субквадратичной сложности — это важно для планов и ортофотоснимков строительных площадок высокого разрешения.
  - Конкурентоспособное качество на задачах классификации и сегментации, совместимость с распространёнными архитектурами детекции и сегментации.

- **W (Weaknesses, слабые стороны):**  
  - Архитектура не решает напрямую временной аспект (динамика строительства во времени).

- **O (Opportunities, возможности):**  
  - Использование в качестве универсальной базовой архитектуры для задач компьютерного зрения в строительстве (контроль прогресса, подсчёт техники, анализ зон риска).
  - Возможность комбинировать с Mamba и другими SSM‑архитектурами для учёта временной динамики.

- **T (Threats, угрозы):**  
  - Конкуренция со стороны специализированных vision‑SSM и улучшенных трансформерных архитектур (ViT, ConvNeXt и др.).

### Mamba

- **S (Strengths):**  
  - Линейная сложность и способность моделировать крайне длинные последовательности.
  - Универсальность по типам данных (текст, временные ряды, последовательности изображений), что важно для комплексного анализа строительных процессов.

- **W (Weaknesses):**  
  - Отсутствие встроенной двумерной структуры, необходимость дополнительного проектирования для пространственных задач.
  - Нет готовой реализации базовой сети для MMSegmentation, высокие инженерные затраты на внедрение.

- **O (Opportunities):**  
  - Построение пространственно‑временных моделей строительства: использование Mamba‑головы поверх признаков UNet++ или HyenaPixel для прогнозирования прогресса работ.
  - Возможность объединять текстовые документы (сметы, планы) и визуальные данные в единой архитектуре.

- **T (Threats):**  
  - Быстрое развитие специализированных vision‑SSM‑архитектур, уже адаптированных к двумерным данным.

### UNet++

- **S (Strengths):**  
  - Проверенная архитектура для семантической сегментации, особенно при наличии объектов разных масштабов.
  - Простота интеграции в MMSegmentation и наличие большого числа готовых метрик и конвейеров для задач, подобных Cityscapes, близких к строительным сценариям.

- **W (Weaknesses):**  
  - Ограниченный глобальный контекст по сравнению с HyenaPixel и Mamba.
  - Масштабируемость по разрешению хуже, чем у современных архитектур с субквадратичным или линейным механизмом внимания/замены внимания.

- **O (Opportunities):**  
  - Быстрый запуск прототипа для строительных задач за счёт перенастройки уже существующих конвейеров разметки городских сцен под собственные классы («здание», «краны», «ограждения» и т.п.).
  - Использование в качестве опорной модели и/или компонента ансамбля.

- **T (Threats):**  
  - Постепенное вытеснение более современными базовыми архитектурами в задачах, требующих очень высокого разрешения и богатого глобального контекста.

## 6. ТЕХНИЧЕСКИЙ СРАВНИТЕЛЬНЫЙ АНАЛИЗ

### Таблица 3. Сравнение технических характеристик и метрик MMSegmentation

| Характеристика / метрика | HyenaPixel | Mamba | UNet++ |
|--------------------------|-----------|-------|--------|
| Архитектура | MetaFormer с двумерным модулем Hyena | Селективная модель в пространстве состояний | Сверточный энкодер–декодер с вложенными пропусками |
| Сложность по размеру входа | Субквадратичная по числу пикселей | O(L) по длине последовательности | Сопоставима с U‑Net |
| Эффективное рецептивное поле | Глобальное, ядра до 191×191 | Глобальное по последовательности | Зависит от глубины и размеров свёрток |
| Наличие готовых конфигураций в mmseg | Требуется собственная реализация базовой сети | Отсутствуют, возможна только пользовательская интеграция | Есть пример U‑Net; UNet++ несложно добавить |
| Базовые метрики в mmseg для сегментации | aAcc, mAcc, mIoU, mDice, mFscore, mPrecision, mRecall | Те же (через IoUMetric) при интеграции | Те же, активно используются как базовые |
| Типичный домен | Задачи компьютерного зрения высокого разрешения, ADE20K, COCO | Язык, аудио, длинные последовательности | Медицинская сегментация, другие двумерные сцены |

## 7. UX/UI АНАЛИЗ (если применимо)

## 8. ВЫЯВЛЕННЫЕ ПРОБЛЕМЫ И ОГРАНИЧЕНИЯ СУЩЕСТВУЮЩИХ РЕШЕНИЙ

### 8.1. Проблема 1: Ограничения по разрешению и глобальному контексту

- Классические U‑Net и UNet++ при увеличении разрешения сталкиваются с ростом потребления памяти, а эффективное рецептивное поле, хотя и велико, всё же ограничено глубиной сети и размерами свёрток; это может быть критично для очень крупных строительных планов и панорамных видов строительных площадок.
- Архитектуры на основе механизма самовнимания (ViT) обеспечивают глобальный контекст, но их квадратичная сложность делает применение к сценам высокого разрешения дорогостоящим; именно это ограничение пытаются преодолеть HyenaPixel и архитектуры семейства Mamba.

### 8.2. Проблема 2: Пространственно‑временное моделирование

- Для строительных задач важно не только текущее состояние (сегментация по одному снимку), но и динамика: изменение объёмов работ, выявление отстающих участков, рост рисков.
- Ни одна из рассматриваемых архитектур в «чистом» виде не решает эту задачу полностью: HyenaPixel и UNet++ сосредоточены на анализе отдельного двумерного кадра, Mamba ориентирована на последовательность, но без встроенной двумерной структуры.

## 9. КОНКУРЕНТНЫЕ ПРЕИМУЩЕСТВА РАЗРАБАТЫВАЕМОГО РЕШЕНИЯ

### 9.1. Преимущество 1: Методика применения HyenaPixel к задачам ДЗЗ в строительстве

- **Описание:** разрабатываемое решение представляет собой методику использования архитектуры HyenaPixel для анализа пространственных данных дистанционного зондирования (ортофотоснимки, спутниковые и аэрофотоданные строительных площадок), включая адаптацию архитектуры, интеграцию в MMSegmentation и выбор набора метрик.
- **Обоснование:**  
- По сравнению с UNet++ HyenaPixel обеспечивает более богатый глобальный контекст и лучшую масштабируемость по разрешению, что критично для крупномасштабных сцен ДЗЗ.
- По сравнению с архитектурами на базе моделей в пространстве состояний для компьютерного зрения методика опирается на уже существующие реализации HyenaPixel и экосистему стандартных сегментационных голов, что снижает инженерные риски.

- **Ценность для пользователя:** формализованная методика даёт готовый путь внедрения HyenaPixel в прикладные задачи сегментации строительных объектов по данным ДЗЗ, с чётко определёнными шагами настройки, обучения и оценки качества.

## 10. ЦЕЛЕВАЯ АУДИТОРИЯ

### 10.1. Первичная целевая аудитория

- Инжиниринговые и строительные компании, использующие беспилотники и видеонаблюдение для мониторинга объектов.
- Девелоперы и BIM‑инженеры, заинтересованные в автоматической сегментации планов и ортофотоснимков для оценки прогресса строительства.
- Подразделения технического надзора и охраны труда, для которых важна сегментация опасных зон, путей эвакуации и временных конструкций.

Характеристики аудитории:

- Работа с большими объёмами двумерных данных (фотографии, схемы, планы) и пространственно‑временных рядов;
- Ограниченные вычислительные ресурсы на объекте или в облачной инфраструктуре, необходимость в эффективных моделях;
- Потребность в интерпретируемых метриках (mIoU, Dice, поклассная точность) для формирования доверия к системе.

### 10.2. Вторичная целевая аудитория

- Академические коллективы и исследовательские подразделения, занимающиеся новыми архитектурами для пространственно‑временного анализа.
- Муниципальные структуры и городские службы, использующие семантическую сегментацию для инвентаризации городской застройки и контроля строительства.

## 11. ВЫВОДЫ

### 11.1. Общие выводы по анализу архитектур

- **UNet++** выступает надёжной и понятной опорной архитектурой для двумерной сегментации, что позволяет относительно быстро реализовать рабочий прототип для анализа строительных сцен, опираясь на богатый набор метрик MMSegmentation.
- **HyenaPixel** является перспективной базовой архитектурой для строительных данных высокого разрешения благодаря большому эффективному рецептивному полю и хорошей масштабируемости; она особенно подходит для задач, где критично качество глобального контекста сцены.
- **Mamba** представляет собой универсальное ядро для работы с последовательностями, особенно ценное при построении пространственно‑временных моделей (учёт истории строительства), хотя требует дополнительной инженерной работы для адаптации к двумерному домену.

### 11.2. Обоснование актуальности проекта

Актуальность проекта, основанного на указанной архитектуре, обусловлена несколькими факторами:

- ростом объёмов визуальных и сенсорных данных в строительстве;
- необходимостью автоматизировать контроль и прогнозирование прогресса работ;
- недостаточной представленностью готовых решений, нацеленных именно на  сценарии решения задач ДЗЗ.

### 11.3. Позиционирование разрабатываемого решения

Разрабатываемое решение можно позиционировать как прикладную методику и набор практических рекомендаций по использованию архитектуры HyenaPixel для анализа пространственных и пространственно‑временных данных ДЗЗ в строительстве. Mamba и UNet++ в рамках работы выполняют роль систем‑аналогов, позволяя обосновать выбор HyenaPixel по качеству, масштабируемости и применимости к данным дистанционного зондирования.

### 11.4. Риски и ограничения

- Необходимость значительного объёма размеченных данных по строительным сценам для достижения высоких значений mIoU и mDice.
- Потенциально высокая вычислительная стоимость при работе с высокими разрешениями и длинными временными рядами, несмотря на более эффективные архитектурные решения.

## СПИСОК ИСТОЧНИКОВ

- Spravil et al. HyenaPixel: Longer context for vision with selective state spaces.
- Gu et al. Mamba: Linear-time sequence modeling with selective state spaces.
- Zhou et al. UNet++: A nested U-Net architecture for medical image segmentation.
- MMSegmentation: OpenMMLab semantic segmentation toolbox and benchmark.
